# AI Agent with MCPs using vLLM, Pydantic AI


<a id="step1"></a>

## Step 1: Launch a vLLM Server

Using [vLLM](https://github.com/vllm-project/vllm) as our inference serving engine. vLLM provides many benefits such as fast model execution, extensive list of supported models, easy to use, and best of all it's open-source.



In [36]:
import shutil, subprocess

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU found. Go to Runtime > Change runtime type, choose a GPU (e.g. T4), then run from the top again.")

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

Tesla T4, 15360 MiB, 7.5


Time to start your vLLM server and creating an end-point for your LLM.



```bash

    --served-model-name Qwen3-4B-Instruct-250 \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \

```


In [37]:
!pip install -q uv
!uv venv -q /content/vllm-env
!uv pip install -q --python /content/vllm-env/bin/python vllm

? A virtual environment already exists at `vllm-env`. Do you want to replace it? [y/n] › yes

✔ A virtual environment already exists at `vllm-env`. Do you want to replace it? · yes


Now start the server in the background and wait until it is ready. The first start downloads the model, so it can take several minutes. If it fails, the last lines of the log are printed.

In [69]:
import subprocess, time, requests

MODEL_ID          = "Qwen/Qwen3-4B-Instruct-2507"
SERVED_MODEL_NAME = "Qwen3"


try:
    cap = float(subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True).stdout.split()[0])
except Exception:
    cap = 7.5
dtype = "half" if cap < 8.0 else "auto"

cmd = [
    "/content/vllm-env/bin/vllm", "serve", MODEL_ID,
    "--served-model-name", SERVED_MODEL_NAME,
    "--api-key", "abc-123",
    "--port", "8000",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
    "--trust-remote-code",
    "--gpu-memory-utilization", "0.9",
    "--dtype", dtype,
    "--max-model-len", "16384",
]
server = subprocess.Popen(cmd, stdout=open("vllm_serve.log", "w"), stderr=subprocess.STDOUT)

ready = False
for _ in range(180):                      # wait up to 15 minutes
    if server.poll() is not None:         # the process exited early: something went wrong
        break
    try:
        r = requests.get("http://localhost:8000/v1/models",
                         headers={"Authorization": "Bearer abc-123"}, timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(5)

if ready:
    print(f"vLLM is ready, serving {MODEL_ID} as '{SERVED_MODEL_NAME}'")
else:
    print(open("vllm_serve.log").read()[-3000:])
    raise RuntimeError("vLLM did not start. See the log above (out of memory? try a smaller model or --max-model-len).")

vLLM is ready, serving Qwen/Qwen3-4B-Instruct-2507 as 'Qwen3'


Upon successful launch, server should be accepting incoming traffic through an OpenAI-compatible API. Set some environment variables for our server

In [40]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


We can verify that model is available at the `BASE_URL` we just set by running the following command.

In [41]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

{"object":"list","data":[{"id":"Qwen3","object":"model","created":1789917874,"owned_by":"vllm","root":"Qwen/Qwen3-4B-Instruct-2507","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-80649864fbbee0e8","object":"model_permission","created":1789917874,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

Congratulations, you now just launched a powerful server that can serve any incoming request and allowing you to build amazing applications. Wasn't that easy?🎉

<a id="step2"></a>

## Step 2: Installing Dependencies

We are going to use `Pydantic AI`. Let's install the dependencies:

In [42]:

!pip install "pydantic-ai-slim[mcp,openai]==1.65.0"

<a id="step3"></a>

## Step 3: Create a simple instance of Pydantic-AI Agent



In [43]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel(SERVED_MODEL_NAME, provider=provider)

Creating an instance the `Agent` class from `pydantic_ai`.

In [44]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

 `pydantic_ai` provides multiple ways to run `Agent`. You can learn more about it [here](https://ai.pydantic.dev/agents/#running-agents).

`async` mode is used. Define a helper function that allow the quick test of agent.

In [45]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output

Test the agent by calling this function.

In [46]:
await run_async("What is the capital of France?")

'The capital of France is Paris.'

<a id="step4"></a>

## Step 4: Write a Date/Time Tool for Agent



In [47]:
await run_async("What’s the date today?")

'I can\'t provide the current date because I don\'t have real-time access to the internet or live data. However, you can easily check today\'s date on your device\'s clock, calendar, or by searching "today\'s date" in any search engine. Let me know if you\'d like help with something else! 😊'

In [48]:
from datetime import datetime
from pydantic_ai import Tool
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [49]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

In [50]:
await run_async("What’s the date today?")

"Today's date is September 20, 2026."

<a id="step5"></a>

## Step 5: Replace Date/time Tool with a MCP server



**Why MCP?** MCP servers provide:
-  Standardized API interfaces
-  Reusable across projects
-  Pre-built functionality

Replace custom time tool with an official MCP time server:

### Installing Time MCP Server


In [51]:
!pip install -q mcp-server-time

In [52]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=Asia/Karachi",
    ],
)

In [53]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

In [54]:
import sys, functools
import pydantic_ai.mcp as _pydantic_mcp
from mcp.client import stdio as _mcp_stdio

_pydantic_mcp.stdio_client = functools.partial(_mcp_stdio.stdio_client, errlog=sys.__stderr__)

In [55]:
await run_async("What’s the date today?")

"Today's date is Sunday, September 20, 2026, in Karachi."

<a id="step6"></a>

## Step 6: Airbnb finder

npx to launch out next server. Therefore, install the required dependencies.

In [76]:
%%bash

if command -v npx >/dev/null 2>&1; then
  echo "Node.js is already installed."
else
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi

Node.js is already installed.


Verify `npm` and `npx` installation:

In [77]:
!node -v && npm -v && npx --version

v20.19.0
10.8.2
10.8.2


In [78]:
%%bash
timeout 180 npx -y @openbnb/mcp-server-airbnb --ignore-robots-txt < /dev/null 2>&1 | tail -2

  "robotsRespected": false
}


In [79]:
import json

MAX_TOOL_CHARS = 8000   # roughly 2-3k tokens per tool result

async def limit_tool_output(ctx, call_tool, name, tool_args):
    """Call the MCP tool, but cut very long results down to MAX_TOOL_CHARS."""
    result = await call_tool(name, tool_args)
    text = result if isinstance(result, str) else json.dumps(result, default=str)
    if len(text) <= MAX_TOOL_CHARS:
        return result
    return text[:MAX_TOOL_CHARS] + "\n...[output truncated to fit the model's context window]"

In [90]:
airbnb_server = MCPServerStdio(
    "npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
    timeout=60,
    process_tool_call=limit_tool_output,
)

In [91]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server],
    system_prompt=system_prompt,
)

In [92]:
await run_async(
    "Find Airbnb stays in Vancouver for 2 adults "
    "from 2026-09-23 to 2026-09-26."
)

"Here are the top Airbnb stays in Vancouver for 2 adults from September 23 to September 26, 2026:\n\n1. **Kriszta's Lovely Little Hideaway**  \n   - Location: 49.27166, -123.06616  \n   - Type: 1 bedroom, 2 queen beds, 1 bath  \n   - Rating: 4.82 (519 reviews)  \n   - Price: $356 for 3 nights (originally $489)  \n   - [View Listing](https://www.airbnb.com/rooms/3678989)  \n\n2. **Clean & Modern 1 Bedroom + Den in Prime Location**  \n   - Location: 49.2768, -123.1271  \n   - Type: 1 bedroom, 2 beds, 1 bath  \n   - Rating: 4.9 (129 reviews)  \n   - Price: $858 for 3 nights  \n   - [View Listing](https://www.airbnb.com/rooms/563248180261220836)  \n\n3. **Heart of Downtown - AC/w parking**  \n   - Location: 49.28, -123.1266  \n   - Type: 2 bedrooms, 3 beds, 2 baths  \n   - Rating: 4.91 (79 reviews)  \n   - Price: $1,196 for 3 nights (originally $1,329)  \n   - [View Listing](https://www.airbnb.com/rooms/1194330887251303443)  \n\n4. **Private 1-bedroom gem in beautiful Dunbar**  \n   - Loca

In [82]:
await run_async("Find a place to stay in Vancouver for next Sunday for 3 nights for 2 adults?")

"Here are some great options for a 3-night stay in Vancouver for 2 adults, next Sunday (September 27, 2026):\n\n1. **Laneway house Vancouver**  \n   - Price: $685 for 3 nights (originally $780)  \n   - Features: 2 bedrooms, 2 beds, 3.5 baths  \n   - Rating: 4.98/5 (40 reviews)  \n   - Location: Latitude 49.2858, Longitude -123.04905  \n   - [View listing](https://www.airbnb.com/rooms/1588738850214570515)\n\n2. **The Dunbar Den | 2BR Retreat w/ Fast WiFi | Parking**  \n   - Price: $514 for 3 nights (originally $657)  \n   - Features: 2 bedrooms, 2 queen beds, 1 bath  \n   - Rating: 5.0/5 (26 reviews)  \n   - Location: Latitude 49.24046, Longitude -123.18436  \n   - [View listing](https://www.airbnb.com/rooms/1632886175386630869)\n\n3. **Villa 21: Cozy & Contemporary Laneway Home**  \n   - Price: $651 for 3 nights (originally $763)  \n   - Features: 1 bedroom, 2 beds, 2 baths  \n   - Rating: 4.99/5 (73 reviews)  \n   - Location: Latitude 49.2534, Longitude -123.1695  \n   - [View listing

<a id="step7"></a>

## Step 7: Expand the Agent



1. Launch weather MCP server
2. Add to agent's tools
3. Make agent suggest best travel dates based on weather

In [83]:
weather_server = MCPServerStdio(
    "npx", args=["-y", "@dangahagan/weather-mcp@1.31.3"],
    timeout=90,   # allow time for the first download
    process_tool_call=limit_tool_output,
)


In [84]:
async with weather_server:
    for tool in await weather_server.list_tools():
        print("-", tool.name)

- get_weather_summary
- get_forecast
- get_current_conditions
- get_alerts
- search_location
- check_service_status


**2. Add it to the agent's tools.** We keep the time and Airbnb servers and add the weather server, and we update the system prompt so the agent knows what it can do.

In [93]:
system_prompt = """
You are a travel planning assistant.

You have three groups of tools:

1. Time tools:
   - get_current_time
   - convert_time

2. Weather tools:
   - search_location
   - get_forecast
   - get_current_conditions
   - get_alerts
   - get_weather_summary

3. Airbnb tools:
   - airbnb_search
   - airbnb_listing_details

IMPORTANT WORKFLOW:

Step 1:
Always call get_current_time first.

Step 2:
Use search_location to find the coordinates for the destination.

Step 3:
Use get_forecast to examine the available weather forecast.

Step 4:
Choose ONE exact 3-night travel window within the requested date range.

Step 5:
Before calling Airbnb, explicitly determine:
- check-in date
- check-out date
- number of adults

The Airbnb dates MUST EXACTLY match the 3-night window selected from the weather forecast.

For a 3-night stay:
check-out date = check-in date + 3 days.

Do NOT change, shift, or independently generate the Airbnb dates after selecting the weather window.

Step 6:
Call airbnb_search using exactly those selected dates and the requested number of adults.

Step 7:
Explain which weather facts influenced the selected dates, then show the Airbnb results.

Never invent dates outside the requested range.
Never use past dates when the user requests future dates.
Use dates in YYYY-MM-DD format for Airbnb.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server, weather_server],
    system_prompt=system_prompt,
)

**3. Make the agent suggest the best travel dates based on the weather.**

In [94]:
await run_async(
    "I want to visit Vancouver for 3 nights with 2 adults sometime in the next two weeks..."
)

'### Selected Travel Dates\nI have selected a 3-night stay in Vancouver from **September 23, 2026, to September 26, 2026**.\n\n### Weather Factors Influencing the Selection\nThe chosen dates were selected based on the following weather considerations:\n- **Rainfall pattern:** The forecast shows increasing precipitation starting on Wednesday, September 23, with a 42% chance of slight rain. While this is not severe, it indicates a period of variable weather.\n- **Temperature stability:** The temperatures during this window are mild and stable, ranging from 52°F to 62°F, which is comfortable for a 3-night stay.\n- **Wind conditions:** Wind speeds are moderate (up to 12 mph), with gusts reaching 22 mph on Monday, September 21, but these are not expected during the selected dates.\n- **No active alerts:** There are no weather alerts or safety concerns for the area during this period.\n- **Daylight duration:** The days are shortening slightly, with sunset times decreasing from 7:14 PM on Sep